# ECSC Developmental Analysis v4.2: Larger-Model Robustness Check

**Model:** Llama-3.1-8B-Instruct (bf16 or 4-bit quantized)

**Purpose:** Verify that key developmental findings replicate with a different, larger LM:
1. Half-life increases with age (developmental coherence)
2. Baseline long-range influence increases with age
3. Anchor specificity remains distributed (not retrieval-like)

**Comparison Model:** Mistral-7B-v0.1 (original analysis)

All analysis parameters are **identical** to v4.2 Mistral runs.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

class Config:
    # === Data ===
    MIN_WORDS = 200

    # === End-of-doc analysis (unchanged from v3) ===
    MIN_TARGET_SIZE = 20
    TARGET_FRACTION = 0.10
    MAX_CONTEXT_FIXED = 512
    MAX_CONTEXT_LONG = 1024
    BENEFIT_EPS = 1.0

    # === Burstiness analysis (v4.2: dual-L + baseline decomposition) ===
    K_TARGETS = 20                    # Number of target positions per doc
    TARGET_START_FRAC = 0.15          # Minimum start position (fraction of doc)
    TARGET_END_FRAC = 0.90            # Maximum end position (fraction of doc)
    TARGET_REGION_TOKENS = 40         # Fixed target size for burstiness

    # Context grid for burstiness (includes 512 for primary analysis)
    CONTEXT_GRID_BURST = [32, 64, 128, 192, 256, 384, 512]

    # Context grid for half-life trace (richer, optional)
    CONTEXT_GRID_HALFLIFE = [4, 8, 12, 16, 20, 24, 28, 32, 64, 128, 256]
    COMPUTE_HALFLIFE_TRACE = False    # Set True for richer analysis

    # Long-range influence computation
    # I_256 = full coverage, I_512 = eligible subset
    SHORT_CONTEXT = 32                # Baseline for I_L computation
    LONG_CONTEXTS = [512, 256]        # Both computed; N reported separately

    # Spike detection
    SPIKE_PERCENTILE = 95
    SPIKE_MASS_TOP_FRAC = 0.10        # Top fraction for spike_mass
    # Note: top_k = max(2, ceil(frac * K)) to behave well with small K

    # Filtering
    MIN_TARGETS_PER_DOC = 8           # For I_256 (full coverage)
    MIN_TARGETS_512 = 4               # Minimum for valid I_512 analysis
    
    # Bootstrap
    N_BOOTSTRAP = 1000
    CI_LEVEL = 0.95

    # Plotting
    MIN_DOCS_FOR_PLOT = 10

config = Config()
print(f"Burstiness analysis v4.2: K={config.K_TARGETS} targets per doc")
print(f"Target region: {config.TARGET_REGION_TOKENS} tokens (fixed)")
print(f"Target range: {config.TARGET_START_FRAC:.0%} - {config.TARGET_END_FRAC:.0%} of doc")
print(f"Context grid: {config.CONTEXT_GRID_BURST}")
print(f"\nDual-L analysis:")
print(f"  I_256: full coverage (t >= 256), min {config.MIN_TARGETS_PER_DOC} targets")
print(f"  I_512: eligible subset (t >= 512), min {config.MIN_TARGETS_512} targets")
print(f"  lo = max({config.TARGET_START_FRAC}*T, L), hi = min({config.TARGET_END_FRAC}*T, T-{config.TARGET_REGION_TOKENS})")
print(f"\nSpike mass: top_k = max(2, ceil({config.SPIKE_MASS_TOP_FRAC}*K))")

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

# Define paths
DRIVE_ROOT = '/content/drive/MyDrive/LRTIA'
DATA_DIR = f'{DRIVE_ROOT}/data'
RESULTS_DIR = f'{DRIVE_ROOT}/results/ecsc'

# Model and version info
MODEL_TAG = 'llama8b'  # Short tag for filenames
VERSION = 'v4.2'
VERSION_DIR = f'{RESULTS_DIR}/{VERSION}_{MODEL_TAG}'
FIGURES_DIR = f'{VERSION_DIR}/figures'

# Create directories if needed
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VERSION_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Model: Llama-3.1-8B-Instruct")
print(f"Model tag: {MODEL_TAG}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {VERSION_DIR}")

In [ ]:
# ============================================================
# LOAD DATA FROM GOOGLE DRIVE
# ============================================================

DATA_FILE = f'{DATA_DIR}/ecsc_transcripts.jsonl'

if os.path.exists(DATA_FILE):
    print(f"Loading data from: {DATA_FILE}")
else:
    # Fallback: upload if not found in Drive
    from google.colab import files
    print("Data file not found in Drive. Please upload transcripts.jsonl:")
    uploaded = files.upload()
    uploaded_name = list(uploaded.keys())[0]
    # Copy to Drive for future use
    import shutil
    shutil.copy(uploaded_name, DATA_FILE)
    print(f"Saved to Drive: {DATA_FILE}")

In [ ]:
# Load data
records = []
with open(DATA_FILE) as f:
    for line in f:
        record = json.loads(line)
        record['pop'] = json.loads(record['population'])
        
        # Text statistics for cross-model comparability
        text = record['text']
        record['n_chars'] = len(text)
        record['n_words'] = len(text.split())
        record['word_count'] = record['n_words']  # Keep for compatibility
        record['age_months'] = record['pop']['age_months']
        records.append(record)

df_all = pd.DataFrame(records)
df = df_all[df_all['word_count'] >= config.MIN_WORDS].copy()

def age_bin(age):
    if age < 72: return '4-6yr'
    elif age < 96: return '6-8yr'
    elif age < 120: return '8-10yr'
    else: return '10+yr'

df['age_group'] = df['age_months'].apply(age_bin)

print(f"Documents: {len(df)} (of {len(df_all)} total)")
print(f"\nBy age group:")
for ag in ['4-6yr', '6-8yr', '8-10yr', '10+yr']:
    n = len(df[df['age_group'] == ag])
    if n > 0:
        print(f"  {ag}: n={n}")

print(f"\nText statistics:")
print(f"  n_chars: {df['n_chars'].median():.0f} median (IQR: {df['n_chars'].quantile(0.25):.0f}-{df['n_chars'].quantile(0.75):.0f})")
print(f"  n_words: {df['n_words'].median():.0f} median (IQR: {df['n_words'].quantile(0.25):.0f}-{df['n_words'].quantile(0.75):.0f})")

In [ ]:
# ============================================================
# LOAD MODEL: Llama-3.1-8B-Instruct
# ============================================================

# Primary: Llama-3.1-8B-Instruct
# Fallback: Qwen2.5-7B-Instruct if Llama unavailable

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_DISPLAY = "Llama-3.1-8B-Instruct"

# Try bf16 first, fall back to 4-bit if OOM
try:
    print(f"Attempting bf16 load of {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    QUANTIZATION = "bf16"
    print(f"Loaded {MODEL_NAME} in bf16")
    
except Exception as e:
    print(f"bf16 failed ({e}), trying 4-bit quantization...")
    
    # Try Llama with 4-bit
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        QUANTIZATION = "4bit-nf4"
        print(f"Loaded {MODEL_NAME} in 4-bit")
        
    except Exception as e2:
        print(f"Llama failed ({e2}), falling back to Qwen2.5-7B-Instruct...")
        
        # Fallback: Qwen2.5-7B-Instruct
        MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
        MODEL_DISPLAY = "Qwen2.5-7B-Instruct"
        MODEL_TAG = "qwen7b"  # Update tag
        VERSION_DIR = f'{RESULTS_DIR}/{VERSION}_{MODEL_TAG}'
        FIGURES_DIR = f'{VERSION_DIR}/figures'
        os.makedirs(VERSION_DIR, exist_ok=True)
        os.makedirs(FIGURES_DIR, exist_ok=True)
        
        try:
            tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True
            )
            QUANTIZATION = "bf16"
        except:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True
            )
            QUANTIZATION = "4bit-nf4"
        
        print(f"Loaded fallback: {MODEL_NAME}")

model.eval()

# Store model info for logging
MODEL_INFO = {
    'model_name': MODEL_NAME,
    'model_display': MODEL_DISPLAY,
    'model_tag': MODEL_TAG,
    'quantization': QUANTIZATION,
}
print(f"\nModel: {MODEL_DISPLAY}")
print(f"Quantization: {QUANTIZATION}")
print(f"Output dir: {VERSION_DIR}")

In [ ]:
# ============================================================
# CORE FUNCTIONS (unchanged from v3)
# ============================================================

@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    """Compute perplexity on tokens in [target_start, target_end]."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    
    total_loss = 0.0
    count = 0
    
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        target_token = token_ids[i + 1]
        token_loss = -log_probs[target_token].item()
        total_loss += token_loss
        count += 1
    
    if count == 0:
        return float('inf'), 0
    
    return np.exp(total_loss / count), total_loss / count


def compute_cumulative_min(contexts, perplexities):
    """Convert to monotone non-increasing curve using cumulative minimum."""
    order = np.argsort(contexts)
    contexts = np.array(contexts)[order]
    perplexities = np.array(perplexities)[order]
    ppl_mon = np.minimum.accumulate(perplexities)
    return contexts, ppl_mon


def interpolate_perplexity(contexts, perplexities, target_ctx):
    """Linearly interpolate perplexity at a target context length."""
    if target_ctx <= contexts[0]:
        return perplexities[0]
    if target_ctx >= contexts[-1]:
        return perplexities[-1]
    
    for i in range(len(contexts) - 1):
        if contexts[i] <= target_ctx <= contexts[i+1]:
            frac = (target_ctx - contexts[i]) / (contexts[i+1] - contexts[i])
            return perplexities[i] + frac * (perplexities[i+1] - perplexities[i])
    
    return perplexities[-1]

In [ ]:
# ============================================================
# BURSTINESS: TARGET SELECTION (per-target eligibility)
# ============================================================

def select_target_positions_for_L(n_tokens, config, L):
    """
    Select K evenly-spaced target positions eligible for context length L.
    
    Eligibility per target:
        t >= L  (enough context for I_L)
        t <= T - target_span  (target fits in doc)
    
    Range: lo = max(0.15*T, L), hi = min(0.9*T, T - target_span)
    
    Args:
        n_tokens: Total tokens in document
        config: Configuration object
        L: Required context length (e.g., 256 or 512)
    
    Returns list of target start positions (token indices).
    """
    # Compute eligible range
    lo = max(int(config.TARGET_START_FRAC * n_tokens), L)
    hi = min(int(config.TARGET_END_FRAC * n_tokens), n_tokens - config.TARGET_REGION_TOKENS)
    
    # Need valid range
    if hi <= lo:
        return []
    
    # Evenly space K targets
    if config.K_TARGETS == 1:
        positions = [(lo + hi) // 2]
    else:
        step = (hi - lo) / (config.K_TARGETS - 1)
        positions = [int(lo + i * step) for i in range(config.K_TARGETS)]
    
    return positions


def get_all_target_positions(n_tokens, config):
    """
    Get target positions for each L value.
    
    Returns dict: {L: [positions]}
    """
    positions_by_L = {}
    for L in config.LONG_CONTEXTS:
        positions_by_L[L] = select_target_positions_for_L(n_tokens, config, L)
    return positions_by_L


# Test target selection
print("Target selection examples:")
for T in [600, 800, 1000, 1200]:
    pos_512 = select_target_positions_for_L(T, config, 512)
    pos_256 = select_target_positions_for_L(T, config, 256)
    print(f"  T={T}: I_256 has {len(pos_256)} targets, I_512 has {len(pos_512)} targets")
    if len(pos_512) > 0:
        print(f"         I_512 range: [{pos_512[0]}, {pos_512[-1]}]")

In [ ]:
# ============================================================
# BURSTINESS: LONG-RANGE INFLUENCE TRACE (I_L) - DUAL L with per-target eligibility
# ============================================================

def compute_influence_at_target(full_tokens, target_pos, context_grid, target_size):
    """
    Compute perplexity at a single target position for multiple context lengths.
    
    Returns dict: {context_length: perplexity}
    """
    results = {}
    
    target_end = target_pos + target_size
    if target_end > len(full_tokens):
        return results
    
    for ctx_len in context_grid:
        # Check if enough context available
        if ctx_len > target_pos:
            continue
        
        # Extract context + target
        start_pos = target_pos - ctx_len
        tokens_to_score = full_tokens[start_pos:target_end]
        
        # Target region within this subsequence
        target_start_local = ctx_len
        target_end_local = ctx_len + target_size
        
        ppl, _ = compute_perplexity_on_region(tokens_to_score, target_start_local, target_end_local)
        results[ctx_len] = ppl
    
    return results


def compute_I_L(ppl_dict, short_ctx, long_ctx):
    """
    Compute long-range influence: I_L = ppl(short) - ppl(long)
    
    Higher I_L means more benefit from long context at this target.
    """
    if short_ctx not in ppl_dict or long_ctx not in ppl_dict:
        return np.nan
    
    return ppl_dict[short_ctx] - ppl_dict[long_ctx]


def compute_burstiness_traces_dual(full_tokens, config):
    """
    Compute long-range influence traces for multiple L values with per-target eligibility.
    
    I_256: computed at all targets where t >= 256 (full coverage)
    I_512: computed at targets where t >= 512 (eligible subset)
    
    Returns:
        traces_by_L: dict of {L: {'positions': [...], 'I_L': [...], 'n_valid': int}}
        all_trace_records: list of per-target records for CSV output
    """
    n_tokens = len(full_tokens)
    
    # Get target positions for each L
    positions_by_L = get_all_target_positions(n_tokens, config)
    
    # Initialize results
    traces_by_L = {}
    all_trace_records = []
    
    for L in config.LONG_CONTEXTS:
        positions = positions_by_L[L]
        I_L_trace = []
        valid_positions = []
        
        for pos in positions:
            ppl_dict = compute_influence_at_target(
                full_tokens, pos, 
                config.CONTEXT_GRID_BURST, 
                config.TARGET_REGION_TOKENS
            )
            
            # Compute I_L for this context length
            I_L = compute_I_L(ppl_dict, config.SHORT_CONTEXT, L)
            
            if not np.isnan(I_L):
                I_L_trace.append(I_L)
                valid_positions.append(pos)
        
        traces_by_L[L] = {
            'positions': valid_positions,
            'I_L': I_L_trace,
            'n_valid': len(I_L_trace)
        }
    
    # Create unified trace records (using I_256 positions as base, adding I_512 where available)
    # Use the L with more targets (256) as the primary set
    primary_L = 256
    primary_positions = traces_by_L[primary_L]['positions']
    
    for i, pos in enumerate(primary_positions):
        record = {
            'target_idx': i,
            'target_pos': pos,
            'target_pos_frac': pos / n_tokens,
        }
        
        # Add I_L for each context length
        for L in config.LONG_CONTEXTS:
            # Find matching position in this L's trace
            if pos in traces_by_L[L]['positions']:
                idx = traces_by_L[L]['positions'].index(pos)
                record[f'I_{L}'] = traces_by_L[L]['I_L'][idx]
            else:
                record[f'I_{L}'] = np.nan
        
        all_trace_records.append(record)
    
    return traces_by_L, all_trace_records

In [ ]:
# ============================================================
# BURSTINESS: METRICS WITH BASELINE+SPIKES DECOMPOSITION
# ============================================================

def compute_burstiness_metrics_decomposed(I_L_trace, config):
    """
    Compute burstiness metrics with baseline+spikes decomposition.
    
    Separates:
    - baseline: mean long-range influence (distributed coherence)
    - spike metrics: computed on CENTERED trace (event-like dependence)
    
    Returns dict with:
        Raw metrics:
        - I_L_mean (baseline), I_L_max, I_L_std, n_targets
        
        Centered spike metrics (on trace - baseline):
        - spike_mass_centered: concentration of positive excursions in top k
        - spike_count_centered: number above 95th percentile of centered trace
        - cv_centered: CV of centered trace
        - kurtosis_centered: excess kurtosis of centered trace
    """
    trace = np.array(I_L_trace)
    trace = trace[~np.isnan(trace)]
    K = len(trace)
    
    if K < 3:
        return {
            # Raw
            'I_L_mean': np.nan,
            'I_L_max': np.nan,
            'I_L_std': np.nan,
            'n_targets': K,
            # Centered spike metrics
            'spike_mass_centered': np.nan,
            'spike_count_centered': np.nan,
            'cv_centered': np.nan,
            'kurtosis_centered': np.nan,
            # Legacy (for compatibility)
            'spike_mass': np.nan,
            'spike_count': np.nan,
            'cv': np.nan,
            'kurtosis': np.nan,
        }
    
    # === BASELINE: mean long-range influence ===
    baseline = np.mean(trace)
    max_val = np.max(trace)
    std_val = np.std(trace)
    
    # === CENTERED TRACE: for spike analysis ===
    trace_centered = trace - baseline
    
    # Top k for spike mass: max(2, ceil(frac * K)) to behave well with small K
    n_top = max(2, int(np.ceil(K * config.SPIKE_MASS_TOP_FRAC)))
    
    # CV on raw trace (for compatibility)
    cv_raw = std_val / abs(baseline) if abs(baseline) > 0.01 else np.nan
    
    # Kurtosis on raw trace
    kurtosis_raw = stats.kurtosis(trace, fisher=True) if std_val > 0 else np.nan
    
    # Spike mass on raw trace (for compatibility)
    positive_trace = trace[trace > 0]
    if len(positive_trace) > 0:
        total_positive = np.sum(positive_trace)
        sorted_trace = np.sort(trace)[::-1]
        top_sum = np.sum(sorted_trace[:n_top])
        spike_mass_raw = top_sum / total_positive if total_positive > 0 else 0
    else:
        spike_mass_raw = 0
    
    # Spike count on raw trace
    threshold_raw = np.percentile(trace, config.SPIKE_PERCENTILE)
    spike_count_raw = np.sum(trace > threshold_raw)
    
    # === CENTERED SPIKE METRICS (key new metrics) ===
    
    # Spike mass on positive excursions (above baseline)
    positive_excursions = trace_centered[trace_centered > 0]
    if len(positive_excursions) > 0:
        total_excursion = np.sum(positive_excursions)
        sorted_centered = np.sort(trace_centered)[::-1]
        # Only sum positive values in top k
        top_vals = sorted_centered[:n_top]
        top_sum = np.sum(top_vals[top_vals > 0])
        spike_mass_centered = top_sum / total_excursion if total_excursion > 0 else 0
    else:
        spike_mass_centered = 0
    
    # Spike count on centered trace (above 95th percentile of centered)
    threshold_centered = np.percentile(trace_centered, config.SPIKE_PERCENTILE)
    spike_count_centered = np.sum(trace_centered > threshold_centered)
    
    # CV on centered trace (std of centered / mean of absolute centered)
    std_centered = np.std(trace_centered)
    mean_abs_centered = np.mean(np.abs(trace_centered))
    cv_centered = std_centered / mean_abs_centered if mean_abs_centered > 0.01 else np.nan
    
    # Kurtosis on centered trace
    kurtosis_centered = stats.kurtosis(trace_centered, fisher=True) if std_centered > 0 else np.nan
    
    return {
        # Raw / baseline
        'I_L_mean': baseline,       # THIS IS THE BASELINE
        'I_L_max': max_val,
        'I_L_std': std_val,
        'n_targets': K,
        
        # Centered spike metrics (NEW - event-like dependence)
        'spike_mass_centered': spike_mass_centered,
        'spike_count_centered': spike_count_centered,
        'cv_centered': cv_centered,
        'kurtosis_centered': kurtosis_centered,
        
        # Legacy raw metrics (for compatibility)
        'spike_mass': spike_mass_raw,
        'spike_count': spike_count_raw,
        'cv': cv_raw,
        'kurtosis': kurtosis_raw,
    }

In [ ]:
# ============================================================
# END-OF-DOC ANALYSIS (unchanged from v3)
# ============================================================

def get_context_lengths(max_context):
    """Generate context lengths with dense sampling at short range."""
    lengths = []
    lengths.extend(range(4, min(33, max_context + 1), 4))
    lengths.extend(range(48, min(129, max_context + 1), 16))
    lengths.extend(range(160, max_context + 1, 32))
    return sorted(set(lengths))


def compute_half_life_robust(contexts, perplexities, max_ctx_fixed=None, benefit_eps=1.0):
    """Compute half-life with cumulative-min envelope and fixed max context."""
    contexts, ppl_mon = compute_cumulative_min(contexts, perplexities)
    
    ppl_at_min = ppl_mon[0]
    ctx_min = contexts[0]
    
    max_ctx_available = contexts[-1]
    if max_ctx_fixed is not None:
        max_ctx_used = min(max_ctx_available, max_ctx_fixed)
    else:
        max_ctx_used = max_ctx_available
    
    ppl_at_max = interpolate_perplexity(contexts, ppl_mon, max_ctx_used)
    total_benefit = ppl_at_min - ppl_at_max
    benefit_ok = total_benefit >= benefit_eps
    
    half_life = np.nan
    if benefit_ok:
        target_ppl = ppl_at_min - 0.5 * total_benefit
        for i in range(len(ppl_mon) - 1):
            if contexts[i+1] > max_ctx_used:
                break
            if ppl_mon[i] >= target_ppl >= ppl_mon[i+1]:
                frac = (ppl_mon[i] - target_ppl) / (ppl_mon[i] - ppl_mon[i+1])
                half_life = contexts[i] + frac * (contexts[i+1] - contexts[i])
                break
    
    ppl_at_32 = interpolate_perplexity(contexts, ppl_mon, 32) if 32 <= max_ctx_available else np.nan
    early_drop = ppl_at_min - ppl_at_32 if not np.isnan(ppl_at_32) else np.nan
    early_drop_pct = 100 * early_drop / total_benefit if (benefit_ok and not np.isnan(early_drop)) else np.nan
    
    early_mask = (contexts >= ctx_min) & (contexts <= 32)
    if early_mask.sum() >= 3:
        slope, _, _, _, _ = stats.linregress(contexts[early_mask], ppl_mon[early_mask])
        early_slope = slope
    else:
        early_slope = np.nan
    
    return {
        'ppl_at_min_ctx': ppl_at_min,
        'ppl_at_32': ppl_at_32,
        'ppl_at_max_ctx': ppl_at_max,
        'total_benefit': total_benefit,
        'benefit_ok': benefit_ok,
        'half_life': half_life,
        'early_drop': early_drop,
        'early_drop_pct': early_drop_pct,
        'early_slope': early_slope,
        'max_ctx_available': max_ctx_available,
        'max_ctx_used': max_ctx_used,
    }


def analyze_document_endofdoc(text):
    """Run end-of-doc context ablation (unchanged from v3)."""
    full_tokens = tokenizer.encode(text)
    n_tokens = len(full_tokens)
    
    target_size = max(config.MIN_TARGET_SIZE, int(n_tokens * config.TARGET_FRACTION))
    target_size = min(target_size, n_tokens - 8)
    
    if target_size < config.MIN_TARGET_SIZE:
        return [], {}
    
    max_context = n_tokens - target_size
    context_lengths = get_context_lengths(max_context)
    
    if len(context_lengths) < 3:
        return [], {}
    
    results = []
    for ctx_len in context_lengths:
        doc_start = n_tokens - target_size - ctx_len
        truncated_tokens = full_tokens[doc_start:]
        target_start = len(truncated_tokens) - target_size
        target_end = len(truncated_tokens)
        ppl, loss = compute_perplexity_on_region(truncated_tokens, target_start, target_end)
        results.append({'context_length': ctx_len, 'perplexity': ppl, 'loss': loss})
    
    meta = {'n_tokens': n_tokens, 'target_size': target_size, 'max_context_available': max_context}
    return results, meta

In [ ]:
# ============================================================
# RUN COMBINED ANALYSIS (v4.1: dual-L with per-target eligibility)
# ============================================================

all_results_endofdoc = []  # For end-of-doc curves
doc_metrics = []           # Per-doc metrics (end-of-doc + burstiness)
burst_traces = []          # Per-doc influence traces (per target)

age_groups = ['4-6yr', '6-8yr', '8-10yr', '10+yr']

# Track eligibility
n_valid_256 = 0
n_valid_512 = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    full_tokens = tokenizer.encode(row['text'])
    n_tokens = len(full_tokens)
    
    # --- End-of-doc analysis (v3) ---
    endofdoc_results, endofdoc_meta = analyze_document_endofdoc(row['text'])
    
    if not endofdoc_results or len(endofdoc_results) < 3:
        continue
    
    # Store raw end-of-doc results
    for r in endofdoc_results:
        r['doc_id'] = row['doc_id']
        r['age_months'] = row['age_months']
        r['age_group'] = row['age_group']
        all_results_endofdoc.append(r)
    
    # Compute end-of-doc metrics
    contexts = [r['context_length'] for r in endofdoc_results]
    perplexities = [r['perplexity'] for r in endofdoc_results]
    
    metrics_fixed = compute_half_life_robust(
        contexts, perplexities, 
        max_ctx_fixed=config.MAX_CONTEXT_FIXED,
        benefit_eps=config.BENEFIT_EPS
    )
    
    # --- Burstiness analysis (dual-L with per-target eligibility) ---
    traces_by_L, trace_records = compute_burstiness_traces_dual(full_tokens, config)
    
    # Check validity for each L
    valid_256 = traces_by_L[256]['n_valid'] >= config.MIN_TARGETS_PER_DOC
    valid_512 = traces_by_L[512]['n_valid'] >= config.MIN_TARGETS_512
    
    if valid_256:
        n_valid_256 += 1
    if valid_512:
        n_valid_512 += 1
    
    # Initialize doc metrics dict
    doc_record = {
        'doc_id': row['doc_id'],
        'age_months': row['age_months'],
        'age_group': row['age_group'],
        
        # Text statistics (cross-model comparability)
        'n_chars': row['n_chars'],
        'n_words': row['n_words'],
        'n_tokens_model': n_tokens,  # Model-specific tokenization
        'word_count': row['word_count'],
        'n_tokens': n_tokens,
        
        # End-of-doc metrics (v3)
        'ppl_min_ctx': metrics_fixed['ppl_at_min_ctx'],
        'ppl_32': metrics_fixed['ppl_at_32'],
        'ppl_max_fixed': metrics_fixed['ppl_at_max_ctx'],
        'total_benefit_fixed': metrics_fixed['total_benefit'],
        'benefit_ok_fixed': metrics_fixed['benefit_ok'],
        'half_life_fixed': metrics_fixed['half_life'],
        'early_drop_pct_fixed': metrics_fixed['early_drop_pct'],
        'early_slope': metrics_fixed['early_slope'],
        
        # Burstiness validity (separate for each L)
        'burst_valid_256': valid_256,
        'burst_valid_512': valid_512,
        'burst_n_targets_256': traces_by_L[256]['n_valid'],
        'burst_n_targets_512': traces_by_L[512]['n_valid'],
    }
    
    # Compute metrics for each L value (if valid)
    for L in config.LONG_CONTEXTS:
        suffix = f'_{L}'
        min_targets = config.MIN_TARGETS_512 if L == 512 else config.MIN_TARGETS_PER_DOC
        is_valid = traces_by_L[L]['n_valid'] >= min_targets
        
        if is_valid:
            metrics_L = compute_burstiness_metrics_decomposed(traces_by_L[L]['I_L'], config)
            
            doc_record[f'burst_baseline{suffix}'] = metrics_L['I_L_mean']  # BASELINE
            doc_record[f'burst_I_L_max{suffix}'] = metrics_L['I_L_max']
            doc_record[f'burst_I_L_std{suffix}'] = metrics_L['I_L_std']
            
            # Centered spike metrics (event-like)
            doc_record[f'burst_spike_mass_c{suffix}'] = metrics_L['spike_mass_centered']
            doc_record[f'burst_spike_count_c{suffix}'] = metrics_L['spike_count_centered']
            doc_record[f'burst_cv_c{suffix}'] = metrics_L['cv_centered']
            doc_record[f'burst_kurtosis_c{suffix}'] = metrics_L['kurtosis_centered']
            
            # Legacy raw metrics
            doc_record[f'burst_spike_mass{suffix}'] = metrics_L['spike_mass']
            doc_record[f'burst_cv{suffix}'] = metrics_L['cv']
        else:
            for col in ['burst_baseline', 'burst_I_L_max', 'burst_I_L_std',
                        'burst_spike_mass_c', 'burst_spike_count_c', 
                        'burst_cv_c', 'burst_kurtosis_c',
                        'burst_spike_mass', 'burst_cv']:
                doc_record[f'{col}{suffix}'] = np.nan
    
    # Store traces (if I_256 is valid - our base coverage)
    if valid_256:
        for record in trace_records:
            record['doc_id'] = row['doc_id']
            record['age_months'] = row['age_months']
            record['age_group'] = row['age_group']
            burst_traces.append(record)
    
    doc_metrics.append(doc_record)

results_df = pd.DataFrame(all_results_endofdoc)
metrics_df = pd.DataFrame(doc_metrics)
traces_df = pd.DataFrame(burst_traces)

print(f"\n{'='*60}")
print("PROCESSING SUMMARY (v4.1)")
print(f"{'='*60}")
print(f"Total documents processed: {len(metrics_df)}")
print(f"\nBurstiness eligibility:")
print(f"  Valid for I_256: {n_valid_256} ({100*n_valid_256/len(metrics_df):.1f}%) - full coverage")
print(f"  Valid for I_512: {n_valid_512} ({100*n_valid_512/len(metrics_df):.1f}%) - eligible subset")
print(f"\nTotal target observations: {len(traces_df)}")
print(f"\nBy age group:")
for ag in age_groups:
    n_total = len(metrics_df[metrics_df['age_group'] == ag])
    n_256 = len(metrics_df[(metrics_df['age_group'] == ag) & (metrics_df['burst_valid_256'])])
    n_512 = len(metrics_df[(metrics_df['age_group'] == ag) & (metrics_df['burst_valid_512'])])
    if n_total > 0:
        print(f"  {ag}: I_256={n_256}/{n_total}, I_512={n_512}/{n_total}")

In [ ]:
# ============================================================
# BURSTINESS RESULTS BY AGE GROUP (v4.1: separate N for each L)
# ============================================================

def bootstrap_ci(values, n_bootstrap=1000, ci=0.95, statistic=np.mean):
    """Compute bootstrap CI for a statistic."""
    values = np.array(values)
    values = values[~np.isnan(values)]
    
    if len(values) < 3:
        return np.nan, np.nan, np.nan
    
    rng = np.random.default_rng(42)
    boot_stats = []
    
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_stats.append(statistic(sample))
    
    alpha = (1 - ci) / 2
    ci_low = np.percentile(boot_stats, alpha * 100)
    ci_high = np.percentile(boot_stats, (1 - alpha) * 100)
    
    return statistic(values), ci_low, ci_high


print("=" * 70)
print("BURSTINESS RESULTS BY AGE GROUP (v4.1)")
print("=" * 70)

# Report n_targets distributions by age bin
print("\n--- N_TARGETS DISTRIBUTION BY AGE GROUP ---")
print("\nI_256 targets per doc (median [IQR]):")
for ag in age_groups:
    ag_df = metrics_df[(metrics_df['age_group'] == ag) & (metrics_df['burst_valid_256'])]
    if len(ag_df) > 0:
        vals = ag_df['burst_n_targets_256'].values
        med = np.median(vals)
        q25, q75 = np.percentile(vals, [25, 75])
        print(f"  {ag}: {med:.0f} [{q25:.0f}, {q75:.0f}] (n={len(ag_df)})")

print("\nI_512 targets per doc (median [IQR]):")
for ag in age_groups:
    ag_df = metrics_df[(metrics_df['age_group'] == ag) & (metrics_df['burst_valid_512'])]
    if len(ag_df) > 0:
        vals = ag_df['burst_n_targets_512'].values
        med = np.median(vals)
        q25, q75 = np.percentile(vals, [25, 75])
        print(f"  {ag}: {med:.0f} [{q25:.0f}, {q75:.0f}] (n={len(ag_df)})")

# Compute summary for each L value with SEPARATE filtering
burst_summaries = {}

for L in config.LONG_CONTEXTS:
    valid_col = f'burst_valid_{L}'
    suffix = f'_{L}'
    
    burst_valid_L = metrics_df[metrics_df[valid_col]].copy()
    print(f"\nI_{L}: {len(burst_valid_L)} documents valid")
    
    burst_summary = []
    
    for ag in age_groups:
        ag_df = burst_valid_L[burst_valid_L['age_group'] == ag]
        
        if len(ag_df) < 5:
            continue
        
        result = {'age_group': ag, 'n': len(ag_df), 'L': L}
        
        # N targets median/IQR
        n_targ = ag_df[f'burst_n_targets_{L}'].values
        result['n_targets_median'] = np.median(n_targ)
        result['n_targets_q25'] = np.percentile(n_targ, 25)
        result['n_targets_q75'] = np.percentile(n_targ, 75)
        
        # Baseline (distributed influence)
        mean, ci_low, ci_high = bootstrap_ci(
            ag_df[f'burst_baseline{suffix}'].values,
            n_bootstrap=config.N_BOOTSTRAP,
            ci=config.CI_LEVEL
        )
        result['baseline_mean'] = mean
        result['baseline_ci_low'] = ci_low
        result['baseline_ci_high'] = ci_high
        
        # Centered spike metrics (event-like)
        for metric in ['spike_mass_c', 'cv_c', 'kurtosis_c']:
            mean, ci_low, ci_high = bootstrap_ci(
                ag_df[f'burst_{metric}{suffix}'].values,
                n_bootstrap=config.N_BOOTSTRAP,
                ci=config.CI_LEVEL
            )
            result[f'{metric}_mean'] = mean
            result[f'{metric}_ci_low'] = ci_low
            result[f'{metric}_ci_high'] = ci_high
        
        burst_summary.append(result)
    
    burst_summaries[L] = pd.DataFrame(burst_summary)

# Display results for I_512 (primary)
print(f"\n{'='*50}")
print(f"PRIMARY: I_512 = ppl(32) - ppl(512)")
print(f"{'='*50}")

burst_summary_512 = burst_summaries[512]
if len(burst_summary_512) > 0:
    print("\n--- BASELINE (mean I_512 = distributed coherence) ---")
    for _, row in burst_summary_512.iterrows():
        print(f"{row['age_group']}: {row['baseline_mean']:.2f} "
              f"[{row['baseline_ci_low']:.2f}, {row['baseline_ci_high']:.2f}] "
              f"(n={row['n']}, K={row['n_targets_median']:.0f})")
    
    print("\n--- SPIKE MASS (centered, event-like concentration) ---")
    for _, row in burst_summary_512.iterrows():
        print(f"{row['age_group']}: {row['spike_mass_c_mean']:.3f} "
              f"[{row['spike_mass_c_ci_low']:.3f}, {row['spike_mass_c_ci_high']:.3f}]")
    
    print("\n--- CV (centered, higher = burstier pattern) ---")
    for _, row in burst_summary_512.iterrows():
        if not np.isnan(row['cv_c_mean']):
            print(f"{row['age_group']}: {row['cv_c_mean']:.3f} "
                  f"[{row['cv_c_ci_low']:.3f}, {row['cv_c_ci_high']:.3f}]")
else:
    print("Not enough data for I_512 analysis")

# Display results for I_256 (full coverage)
print(f"\n{'='*50}")
print(f"FULL COVERAGE: I_256 = ppl(32) - ppl(256)")
print(f"{'='*50}")

burst_summary_256 = burst_summaries[256]
if len(burst_summary_256) > 0:
    print("\n--- BASELINE (mean I_256) ---")
    for _, row in burst_summary_256.iterrows():
        print(f"{row['age_group']}: {row['baseline_mean']:.2f} "
              f"[{row['baseline_ci_low']:.2f}, {row['baseline_ci_high']:.2f}] "
              f"(n={row['n']}, K={row['n_targets_median']:.0f})")
    
    print("\n--- SPIKE MASS (centered) ---")
    for _, row in burst_summary_256.iterrows():
        print(f"{row['age_group']}: {row['spike_mass_c_mean']:.3f} "
              f"[{row['spike_mass_c_ci_low']:.3f}, {row['spike_mass_c_ci_high']:.3f}]")

# Use I_512 as primary for downstream visualizations
burst_summary_df = burst_summary_512 if len(burst_summary_512) > 0 else burst_summary_256
burst_valid_df = metrics_df[metrics_df['burst_valid_512']].copy() if len(burst_summary_512) > 0 else metrics_df[metrics_df['burst_valid_256']].copy()

In [ ]:
# ============================================================
# STATISTICAL TESTS FOR BURSTINESS (v4.1)
# ============================================================

print("\n" + "=" * 70)
print("BURSTINESS STATISTICAL TESTS (v4.1)")
print("=" * 70)

# Test both I_512 and I_256
for L in [512, 256]:
    valid_col = f'burst_valid_{L}'
    suffix = f'_{L}'
    
    burst_valid_L = metrics_df[metrics_df[valid_col]].copy()
    
    if len(burst_valid_L) < 20:
        print(f"\nI_{L}: Not enough data for statistical tests")
        continue
    
    print(f"\n{'='*40}")
    print(f"I_{L} (n={len(burst_valid_L)})")
    print(f"{'='*40}")
    
    print(f"\n--- Correlations with Age (Spearman) ---")
    
    test_metrics = [
        (f'burst_baseline{suffix}', 'Baseline (distributed)'),
        (f'burst_spike_mass_c{suffix}', 'Spike mass (centered)'),
        (f'burst_cv_c{suffix}', 'CV (centered)'),
        (f'burst_kurtosis_c{suffix}', 'Kurtosis (centered)'),
    ]
    
    for col, label in test_metrics:
        valid = burst_valid_L[[col, 'age_months']].dropna()
        if len(valid) > 10:
            rho, p = stats.spearmanr(valid['age_months'], valid[col])
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            print(f"{label:<30}: rho={rho:+.3f}, p={p:.4f} {sig}")
    
    print(f"\n--- Kruskal-Wallis (overall age group effect) ---")
    for col, label in test_metrics:
        groups = [burst_valid_L[burst_valid_L['age_group'] == ag][col].dropna().values 
                  for ag in age_groups if len(burst_valid_L[burst_valid_L['age_group'] == ag]) > 0]
        groups = [g for g in groups if len(g) >= 3]
        
        if len(groups) >= 2:
            h, p = stats.kruskal(*groups)
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            print(f"{label:<30}: H={h:.2f}, p={p:.4f} {sig}")

# Compare I_512 vs I_256 baselines (paired test on docs valid for both)
print(f"\n{'='*40}")
print("COMPARISON: I_512 vs I_256")
print(f"{'='*40}")

both_valid = metrics_df[metrics_df['burst_valid_512'] & metrics_df['burst_valid_256']].copy()
print(f"\nDocs valid for both: {len(both_valid)}")

if len(both_valid) > 10:
    valid_both = both_valid[['burst_baseline_512', 'burst_baseline_256']].dropna()
    if len(valid_both) > 10:
        stat, p = stats.wilcoxon(valid_both['burst_baseline_512'], valid_both['burst_baseline_256'])
        diff_mean = (valid_both['burst_baseline_512'] - valid_both['burst_baseline_256']).mean()
        print(f"Mean difference (I_512 - I_256): {diff_mean:.2f}")
        print(f"Wilcoxon signed-rank: p={p:.4f}")

In [ ]:
# ============================================================
# EXAMPLE TRACES: YOUNG vs OLDER DOCUMENT (dual L)
# ============================================================

print("\n" + "=" * 70)
print("EXAMPLE DOCUMENT TRACES")
print("=" * 70)

# Use docs valid for I_512 for examples
burst_valid_512_df = metrics_df[metrics_df['burst_valid_512']].copy()

# Find example docs with good trace data
young_docs = burst_valid_512_df[burst_valid_512_df['age_group'] == '4-6yr'].nlargest(5, 'burst_n_targets_512')
older_docs = burst_valid_512_df[burst_valid_512_df['age_group'].isin(['8-10yr', '10+yr'])].nlargest(5, 'burst_n_targets_512')

if len(young_docs) > 0 and len(older_docs) > 0:
    young_example = young_docs.iloc[0]
    older_example = older_docs.iloc[0]
    
    print(f"\nYoung example: {young_example['doc_id']} (age={young_example['age_months']}mo, n_tokens={young_example['n_tokens']})")
    print(f"  I_512 targets: {young_example['burst_n_targets_512']}, I_256 targets: {young_example['burst_n_targets_256']}")
    print(f"  Baseline I_512: {young_example['burst_baseline_512']:.2f}")
    print(f"  Spike mass (centered): {young_example['burst_spike_mass_c_512']:.3f}")
    
    print(f"\nOlder example: {older_example['doc_id']} (age={older_example['age_months']}mo, n_tokens={older_example['n_tokens']})")
    print(f"  I_512 targets: {older_example['burst_n_targets_512']}, I_256 targets: {older_example['burst_n_targets_256']}")
    print(f"  Baseline I_512: {older_example['burst_baseline_512']:.2f}")
    print(f"  Spike mass (centered): {older_example['burst_spike_mass_c_512']:.3f}")
    
    # Get traces for these docs
    young_trace = traces_df[traces_df['doc_id'] == young_example['doc_id']]
    older_trace = traces_df[traces_df['doc_id'] == older_example['doc_id']]
    
    # Plot with both L values
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Top row: I_512 traces with baseline
    ax = axes[0, 0]
    # Filter to targets with valid I_512
    yt_512 = young_trace[young_trace['I_512'].notna()]
    if len(yt_512) > 0:
        baseline = young_example['burst_baseline_512']
        ax.plot(yt_512['target_pos_frac'], yt_512['I_512'], 'o-', 
                color='#e74c3c', linewidth=2, markersize=6, label='I_512')
        ax.axhline(baseline, color='#e74c3c', linestyle='--', alpha=0.5, label=f'baseline={baseline:.1f}')
        ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
        ax.fill_between(yt_512['target_pos_frac'], baseline, yt_512['I_512'], 
                        where=yt_512['I_512'] > baseline, alpha=0.3, color='#e74c3c')
    ax.set_xlabel('Position in Document (fraction)')
    ax.set_ylabel('I_512')
    ax.set_title(f"Young ({young_example['age_months']}mo): I_512 trace\n"
                 f"baseline={young_example['burst_baseline_512']:.1f}, spike_mass_c={young_example['burst_spike_mass_c_512']:.3f}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    ax = axes[0, 1]
    ot_512 = older_trace[older_trace['I_512'].notna()]
    if len(ot_512) > 0:
        baseline = older_example['burst_baseline_512']
        ax.plot(ot_512['target_pos_frac'], ot_512['I_512'], 'o-',
                color='#27ae60', linewidth=2, markersize=6, label='I_512')
        ax.axhline(baseline, color='#27ae60', linestyle='--', alpha=0.5, label=f'baseline={baseline:.1f}')
        ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
        ax.fill_between(ot_512['target_pos_frac'], baseline, ot_512['I_512'],
                        where=ot_512['I_512'] > baseline, alpha=0.3, color='#27ae60')
    ax.set_xlabel('Position in Document (fraction)')
    ax.set_ylabel('I_512')
    ax.set_title(f"Older ({older_example['age_months']}mo): I_512 trace\n"
                 f"baseline={older_example['burst_baseline_512']:.1f}, spike_mass_c={older_example['burst_spike_mass_c_512']:.3f}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Bottom row: Comparison of I_512 vs I_256
    ax = axes[1, 0]
    ax.plot(young_trace['target_pos_frac'], young_trace['I_256'], 's-', 
            color='#c0392b', linewidth=1.5, markersize=5, alpha=0.7, label='I_256 (full)')
    yt_512 = young_trace[young_trace['I_512'].notna()]
    if len(yt_512) > 0:
        ax.plot(yt_512['target_pos_frac'], yt_512['I_512'], 'o-', 
                color='#e74c3c', linewidth=2, markersize=6, label='I_512 (subset)')
    ax.axvline(512 / young_example['n_tokens'], color='gray', linestyle=':', alpha=0.5, label='pos=512')
    ax.set_xlabel('Position in Document (fraction)')
    ax.set_ylabel('I_L')
    ax.set_title(f"Young: I_512 vs I_256 comparison\n(I_512 valid for t >= 512)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    ax = axes[1, 1]
    ax.plot(older_trace['target_pos_frac'], older_trace['I_256'], 's-',
            color='#1e8449', linewidth=1.5, markersize=5, alpha=0.7, label='I_256 (full)')
    ot_512 = older_trace[older_trace['I_512'].notna()]
    if len(ot_512) > 0:
        ax.plot(ot_512['target_pos_frac'], ot_512['I_512'], 'o-',
                color='#27ae60', linewidth=2, markersize=6, label='I_512 (subset)')
    ax.axvline(512 / older_example['n_tokens'], color='gray', linestyle=':', alpha=0.5, label='pos=512')
    ax.set_xlabel('Position in Document (fraction)')
    ax.set_ylabel('I_L')
    ax.set_title(f"Older: I_512 vs I_256 comparison\n(I_512 valid for t >= 512)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('ecsc_burst_example_traces.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Not enough I_512-valid examples for comparison plot.")
    print("Trying I_256-valid examples...")
    
    burst_valid_256_df = metrics_df[metrics_df['burst_valid_256']].copy()
    young_docs = burst_valid_256_df[burst_valid_256_df['age_group'] == '4-6yr'].nlargest(3, 'burst_n_targets_256')
    older_docs = burst_valid_256_df[burst_valid_256_df['age_group'].isin(['8-10yr', '10+yr'])].nlargest(3, 'burst_n_targets_256')
    
    if len(young_docs) > 0 and len(older_docs) > 0:
        print(f"Found {len(young_docs)} young, {len(older_docs)} older examples with I_256")

In [ ]:
# ============================================================
# BURSTINESS VISUALIZATION (v4.1: separate plots for I_512 and I_256)
# ============================================================

colors = {'4-6yr': '#e74c3c', '6-8yr': '#f39c12', '8-10yr': '#27ae60', '10+yr': '#3498db'}

# Get valid data for each L
burst_valid_512_df = metrics_df[metrics_df['burst_valid_512']].copy()
burst_valid_256_df = metrics_df[metrics_df['burst_valid_256']].copy()

# Use I_512 summary if available, else I_256
primary_L = 512 if len(burst_summaries[512]) > 0 else 256
burst_summary_df = burst_summaries[primary_L]
burst_valid_primary = burst_valid_512_df if primary_L == 512 else burst_valid_256_df

print(f"Primary visualization: I_{primary_L}")

if len(burst_summary_df) == 0:
    print("WARNING: No age groups have enough valid burstiness data for visualization.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f'Burstiness Analysis v4.1: Baseline vs Event-Like (I_{primary_L})', 
                 fontsize=14, fontweight='bold')
    
    suffix = f'_{primary_L}'
    
    # 1. BASELINE by age group
    ax = axes[0, 0]
    x_pos = range(len(burst_summary_df))
    means = burst_summary_df['baseline_mean'].values
    ci_low = burst_summary_df['baseline_ci_low'].values
    ci_high = burst_summary_df['baseline_ci_high'].values
    errors = [means - ci_low, ci_high - means]
    
    bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
                  color=[colors.get(ag, 'gray') for ag in burst_summary_df['age_group']],
                  alpha=0.7, edgecolor='black')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"{row['age_group']}\n(n={row['n']})" for _, row in burst_summary_df.iterrows()])
    ax.set_ylabel(f'Baseline (mean I_{primary_L})')
    ax.set_title(f'BASELINE: Distributed Coherence')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 2. SPIKE MASS (centered)
    ax = axes[0, 1]
    means = burst_summary_df['spike_mass_c_mean'].values
    ci_low = burst_summary_df['spike_mass_c_ci_low'].values
    ci_high = burst_summary_df['spike_mass_c_ci_high'].values
    errors = [means - ci_low, ci_high - means]
    
    bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
                  color=[colors.get(ag, 'gray') for ag in burst_summary_df['age_group']],
                  alpha=0.7, edgecolor='black')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"{row['age_group']}" for _, row in burst_summary_df.iterrows()])
    ax.set_ylabel('Spike Mass (centered)')
    ax.set_title('EVENT-LIKE: Spike Concentration')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 3. CV (centered)
    ax = axes[0, 2]
    means = burst_summary_df['cv_c_mean'].values
    ci_low = burst_summary_df['cv_c_ci_low'].values
    ci_high = burst_summary_df['cv_c_ci_high'].values
    errors = [means - ci_low, ci_high - means]
    
    bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
                  color=[colors.get(ag, 'gray') for ag in burst_summary_df['age_group']],
                  alpha=0.7, edgecolor='black')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"{row['age_group']}" for _, row in burst_summary_df.iterrows()])
    ax.set_ylabel('CV (centered)')
    ax.set_title('EVENT-LIKE: Variability')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 4. Scatter: age vs BASELINE
    ax = axes[1, 0]
    for ag in age_groups:
        ag_data = burst_valid_primary[burst_valid_primary['age_group'] == ag]
        if len(ag_data) > 0:
            ax.scatter(ag_data['age_months'], ag_data[f'burst_baseline{suffix}'],
                       alpha=0.5, label=ag, color=colors.get(ag, 'gray'), s=30)
    
    valid = burst_valid_primary[['age_months', f'burst_baseline{suffix}']].dropna()
    if len(valid) > 10:
        z = np.polyfit(valid['age_months'], valid[f'burst_baseline{suffix}'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(valid['age_months'].min(), valid['age_months'].max(), 100)
        ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2)
    
    ax.set_xlabel('Age (months)')
    ax.set_ylabel(f'Baseline I_{primary_L}')
    ax.set_title('Age vs Baseline')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 5. Scatter: age vs SPIKE MASS
    ax = axes[1, 1]
    for ag in age_groups:
        ag_data = burst_valid_primary[burst_valid_primary['age_group'] == ag]
        if len(ag_data) > 0:
            ax.scatter(ag_data['age_months'], ag_data[f'burst_spike_mass_c{suffix}'],
                       alpha=0.5, label=ag, color=colors.get(ag, 'gray'), s=30)
    
    valid = burst_valid_primary[['age_months', f'burst_spike_mass_c{suffix}']].dropna()
    if len(valid) > 10:
        z = np.polyfit(valid['age_months'], valid[f'burst_spike_mass_c{suffix}'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(valid['age_months'].min(), valid['age_months'].max(), 100)
        ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2)
    
    ax.set_xlabel('Age (months)')
    ax.set_ylabel('Spike Mass (centered)')
    ax.set_title('Age vs Spike Mass')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 6. Box plot: baseline distributions
    ax = axes[1, 2]
    box_data = [burst_valid_primary[burst_valid_primary['age_group'] == ag][f'burst_baseline{suffix}'].dropna().values 
                for ag in age_groups if ag in burst_valid_primary['age_group'].values]
    box_labels = [ag for ag in age_groups if ag in burst_valid_primary['age_group'].values]
    
    if box_data and any(len(d) > 0 for d in box_data):
        bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
        for patch, ag in zip(bp['boxes'], box_labels):
            patch.set_facecolor(colors.get(ag, 'gray'))
            patch.set_alpha(0.7)
    
    ax.set_ylabel(f'Baseline I_{primary_L}')
    ax.set_title('Distribution of Baseline')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('ecsc_burstiness_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Additional figure: I_512 vs I_256 comparison (for docs valid for both)
    both_valid = metrics_df[metrics_df['burst_valid_512'] & metrics_df['burst_valid_256']].copy()
    
    if len(both_valid) >= 10:
        fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
        fig2.suptitle(f'Comparison: I_512 vs I_256 (n={len(both_valid)} docs valid for both)', 
                      fontsize=12, fontweight='bold')
        
        # Paired scatter: baselines
        ax = axes2[0]
        for ag in age_groups:
            ag_data = both_valid[both_valid['age_group'] == ag]
            if len(ag_data) > 0:
                ax.scatter(ag_data['burst_baseline_256'], ag_data['burst_baseline_512'],
                           alpha=0.5, label=ag, color=colors.get(ag, 'gray'), s=30)
        
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
        ax.set_xlabel('Baseline I_256')
        ax.set_ylabel('Baseline I_512')
        ax.set_title('Baselines: I_512 vs I_256')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Difference by age group
        ax = axes2[1]
        both_valid['baseline_diff'] = both_valid['burst_baseline_512'] - both_valid['burst_baseline_256']
        
        box_data = [both_valid[both_valid['age_group'] == ag]['baseline_diff'].dropna().values 
                    for ag in age_groups if ag in both_valid['age_group'].values]
        box_labels = [ag for ag in age_groups if ag in both_valid['age_group'].values]
        
        if box_data and any(len(d) > 0 for d in box_data):
            bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
            for patch, ag in zip(bp['boxes'], box_labels):
                patch.set_facecolor(colors.get(ag, 'gray'))
                patch.set_alpha(0.7)
        
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_ylabel('I_512 - I_256')
        ax.set_title('Additional benefit from 512 vs 256')
        ax.grid(True, alpha=0.3, axis='y')
        
        # N valid for each L by age group
        ax = axes2[2]
        x = np.arange(len(age_groups))
        width = 0.35
        
        n_256 = [len(metrics_df[(metrics_df['age_group'] == ag) & (metrics_df['burst_valid_256'])]) 
                 for ag in age_groups]
        n_512 = [len(metrics_df[(metrics_df['age_group'] == ag) & (metrics_df['burst_valid_512'])]) 
                 for ag in age_groups]
        
        ax.bar(x - width/2, n_256, width, label='I_256', alpha=0.7, color='#3498db')
        ax.bar(x + width/2, n_512, width, label='I_512', alpha=0.7, color='#e74c3c')
        ax.set_xticks(x)
        ax.set_xticklabels(age_groups)
        ax.set_ylabel('N documents')
        ax.set_title('Sample sizes by L')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig('ecsc_burstiness_512_vs_256.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print(f"Only {len(both_valid)} docs valid for both I_512 and I_256 - skipping comparison plot")

In [ ]:
# ============================================================
# END-OF-DOC RESULTS (v3 compatibility)
# ============================================================

print("\n" + "=" * 70)
print("END-OF-DOC RESULTS (v3 metrics)")
print("=" * 70)

endofdoc_summary = []

for ag in age_groups:
    ag_df = metrics_df[metrics_df['age_group'] == ag]
    
    if len(ag_df) < 5:
        continue
    
    result = {'age_group': ag, 'n': len(ag_df)}
    
    for metric in ['half_life_fixed', 'early_slope', 'early_drop_pct_fixed', 'ppl_min_ctx']:
        mean, ci_low, ci_high = bootstrap_ci(
            ag_df[metric].values,
            n_bootstrap=config.N_BOOTSTRAP,
            ci=config.CI_LEVEL
        )
        result[f'{metric}_mean'] = mean
        result[f'{metric}_ci_low'] = ci_low
        result[f'{metric}_ci_high'] = ci_high
    
    endofdoc_summary.append(result)

endofdoc_summary_df = pd.DataFrame(endofdoc_summary)

print("\n--- Half-Life (end-of-doc, fixed max=512) ---")
for _, row in endofdoc_summary_df.iterrows():
    print(f"{row['age_group']}: {row['half_life_fixed_mean']:.1f} "
          f"[{row['half_life_fixed_ci_low']:.1f}, {row['half_life_fixed_ci_high']:.1f}] "
          f"(n={row['n']})")

print("\n--- Early Drop % ---")
for _, row in endofdoc_summary_df.iterrows():
    print(f"{row['age_group']}: {row['early_drop_pct_fixed_mean']:.1f}% "
          f"[{row['early_drop_pct_fixed_ci_low']:.1f}, {row['early_drop_pct_fixed_ci_high']:.1f}]")

In [ ]:
# ============================================================
# SAVE RESULTS TO GOOGLE DRIVE (v4.2 - Robustness)
# ============================================================

# Add model column to metrics_df for easy identification
metrics_df['model'] = MODEL_DISPLAY
metrics_df['model_tag'] = MODEL_TAG

# Per-document metrics (combined)
metrics_path = f'{VERSION_DIR}/ecsc_{VERSION}_{MODEL_TAG}_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

# Burstiness traces (optional - skip if huge)
SAVE_TRACES = False  # Set True to save per-target traces
if SAVE_TRACES:
    traces_df['model'] = MODEL_DISPLAY
    traces_path = f'{VERSION_DIR}/ecsc_{VERSION}_{MODEL_TAG}_traces.csv'
    traces_df.to_csv(traces_path, index=False)

# Burstiness summary by age group (for each L)
for L in config.LONG_CONTEXTS:
    if L in burst_summaries and len(burst_summaries[L]) > 0:
        burst_summaries[L]['model'] = MODEL_DISPLAY
        burst_summaries[L].to_csv(f'{VERSION_DIR}/ecsc_{VERSION}_{MODEL_TAG}_burst_summary_{L}.csv', index=False)

# Primary summary (I_512)
if len(burst_summary_df) > 0:
    burst_summary_df['model'] = MODEL_DISPLAY
    burst_summary_df.to_csv(f'{VERSION_DIR}/ecsc_{VERSION}_{MODEL_TAG}_burst_summary.csv', index=False)

# End-of-doc summary by age group
endofdoc_summary_df['model'] = MODEL_DISPLAY
endofdoc_summary_df.to_csv(f'{VERSION_DIR}/ecsc_{VERSION}_{MODEL_TAG}_endofdoc_summary.csv', index=False)

# Config with model info
config_dict = {
    'VERSION': VERSION,
    'MODEL_NAME': MODEL_NAME,
    'MODEL_DISPLAY': MODEL_DISPLAY,
    'MODEL_TAG': MODEL_TAG,
    'QUANTIZATION': QUANTIZATION,
    'MIN_WORDS': config.MIN_WORDS,
    'K_TARGETS': config.K_TARGETS,
    'TARGET_START_FRAC': config.TARGET_START_FRAC,
    'TARGET_END_FRAC': config.TARGET_END_FRAC,
    'TARGET_REGION_TOKENS': config.TARGET_REGION_TOKENS,
    'CONTEXT_GRID_BURST': config.CONTEXT_GRID_BURST,
    'SHORT_CONTEXT': config.SHORT_CONTEXT,
    'LONG_CONTEXTS': config.LONG_CONTEXTS,
    'SPIKE_PERCENTILE': config.SPIKE_PERCENTILE,
    'SPIKE_MASS_TOP_FRAC': config.SPIKE_MASS_TOP_FRAC,
    'MIN_TARGETS_PER_DOC': config.MIN_TARGETS_PER_DOC,
    'MAX_CONTEXT_FIXED': config.MAX_CONTEXT_FIXED,
    'N_BOOTSTRAP': config.N_BOOTSTRAP,
}
with open(f'{VERSION_DIR}/ecsc_{VERSION}_{MODEL_TAG}_config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)

# Save figures to Drive
import shutil
for fig_name in ['ecsc_burstiness_summary.png', 'ecsc_burstiness_512_vs_256.png', 'ecsc_burst_example_traces.png']:
    if os.path.exists(fig_name):
        # Rename with model tag
        new_name = fig_name.replace('ecsc_', f'ecsc_{MODEL_TAG}_')
        shutil.copy(fig_name, f'{FIGURES_DIR}/{new_name}')

print(f"\n{'='*60}")
print(f"ROBUSTNESS CHECK: {MODEL_DISPLAY}")
print(f"{'='*60}")
print(f"\nSaved to {VERSION_DIR}/:")
print(f"  - ecsc_{VERSION}_{MODEL_TAG}_metrics.csv (per-document)")
if SAVE_TRACES:
    print(f"  - ecsc_{VERSION}_{MODEL_TAG}_traces.csv (per-target)")
print(f"  - ecsc_{VERSION}_{MODEL_TAG}_burst_summary.csv (primary: I_512)")
print(f"  - ecsc_{VERSION}_{MODEL_TAG}_burst_summary_512.csv")
print(f"  - ecsc_{VERSION}_{MODEL_TAG}_burst_summary_256.csv")
print(f"  - ecsc_{VERSION}_{MODEL_TAG}_endofdoc_summary.csv")
print(f"  - ecsc_{VERSION}_{MODEL_TAG}_config.json")
print(f"\nFigures saved to {FIGURES_DIR}/")

In [ ]:
# ============================================================
# OPTIONAL: Download copies to local machine
# ============================================================
# (Results are already saved to Google Drive)

from google.colab import files

download_local = False  # Set to True to download local copies

if download_local:
    files.download(f'{VERSION_DIR}/ecsc_{VERSION}_metrics.csv')
    files.download(f'{VERSION_DIR}/ecsc_{VERSION}_burst_summary.csv')
    files.download(f'{VERSION_DIR}/ecsc_{VERSION}_config.json')
    print("Downloaded local copies")
else:
    print(f"Results saved to Drive: {VERSION_DIR}")
    print("Set download_local = True to download local copies")

## Summary (v4.2)

### Dual-L with Per-Target Eligibility

- **I_256**: Full coverage (all targets where t >= 256)
- **I_512**: Eligible subset (targets where t >= 512), N reported separately

Target selection: `lo = max(0.15*T, L)`, `hi = min(0.9*T, T - 40)`

### Baseline vs Event-Like Decomposition

**BASELINE (distributed coherence):**
- `baseline_L = mean(I_L trace)`
- Higher baseline → more overall benefit from long-range context

**SPIKE METRICS (event-like dependence):**
- Computed on `trace_centered = I_L - baseline`
- `spike_mass_c`: top_k = max(2, ceil(0.1*K)) for robustness
- `cv_c`: coefficient of variation of centered trace
- `kurtosis_c`: excess kurtosis (spiky tails)

### Outputs

| File | Description |
|------|-------------|
| `ecsc_age_v4_metrics.csv` | Per-document (I_512 and I_256, n_targets, baseline, spike metrics) |
| `ecsc_age_v4_traces.csv` | Per-target I_L values |
| `ecsc_age_v4_burst_summary_512.csv` | I_512 by age (n, n_targets median/IQR, baseline, spike metrics) |
| `ecsc_age_v4_burst_summary_256.csv` | I_256 by age (full coverage) |
| `ecsc_burstiness_summary.png` | Baseline vs spike visualization |
| `ecsc_burstiness_512_vs_256.png` | I_512 vs I_256 comparison |

In [ ]:
# ============================================================
# ANCHOR SPECIFICITY: Sliding-Window Ablation
# ============================================================
# 
# For each doc with burst_valid_512:
#   1. Find t* = argmax(I_512) - the target with max long-range influence
#   2. Mask sliding windows within the 512-token context
#   3. Measure impact of each window on prediction
#   4. Compute "anchor specificity" metrics:
#      - top_share: fraction of impact from single best window (high = retrieval-like)
#      - n_eff: effective number of contributing windows (low = concentrated)
#
# This distinguishes:
#   (a) Retrieval-like: benefit from specific earlier span
#   (b) Distributed: benefit spread across many windows

print("=" * 70)
print("ANCHOR SPECIFICITY ANALYSIS")
print("=" * 70)

In [ ]:
# ============================================================
# ANCHOR SPECIFICITY: Configuration
# ============================================================

class AnchorConfig:
    # Sliding window parameters
    WINDOW_SIZE = 32          # Size of masked window
    STRIDE = 16               # Stride between windows (16 = good resolution, 32 = faster)
    CONTEXT_L = 512           # Long context for ablation
    
    # Masking strategy: use a neutral token repeated
    # We'll use the tokenizer's unk_token or a common token
    USE_UNK_MASK = True       # If True, use <unk>; else use period token
    
    # Also compute for L=256 for comparison
    ALSO_DO_256 = True
    CONTEXT_S = 256

anchor_config = AnchorConfig()

# Get mask token
if anchor_config.USE_UNK_MASK and tokenizer.unk_token_id is not None:
    MASK_TOKEN_ID = tokenizer.unk_token_id
else:
    # Use a common neutral token (period)
    MASK_TOKEN_ID = tokenizer.encode('.')[0]

print(f"Anchor specificity config:")
print(f"  Window size: {anchor_config.WINDOW_SIZE}")
print(f"  Stride: {anchor_config.STRIDE}")
print(f"  Context: {anchor_config.CONTEXT_L} (primary), {anchor_config.CONTEXT_S} (secondary)")
print(f"  Mask token ID: {MASK_TOKEN_ID} ('{tokenizer.decode([MASK_TOKEN_ID])}')")

n_windows_512 = len(range(0, anchor_config.CONTEXT_L - anchor_config.WINDOW_SIZE + 1, anchor_config.STRIDE))
print(f"  Windows per doc (L=512): {n_windows_512}")

In [ ]:
# ============================================================
# ANCHOR SPECIFICITY: Core Functions
# ============================================================

def find_max_I_512_target(doc_traces_df, doc_id):
    """Find the target with maximum I_512 for a document."""
    doc_trace = doc_traces_df[doc_traces_df['doc_id'] == doc_id]
    if len(doc_trace) == 0:
        return None, None, None
    
    # Filter to valid I_512 values
    valid = doc_trace[doc_trace['I_512'].notna()]
    if len(valid) == 0:
        return None, None, None
    
    idx_max = valid['I_512'].idxmax()
    t_star = valid.loc[idx_max, 'target_pos']
    I_512_star = valid.loc[idx_max, 'I_512']
    t_star_frac = valid.loc[idx_max, 'target_pos_frac']
    
    return int(t_star), I_512_star, t_star_frac


def compute_ppl_with_masked_window(full_tokens, target_pos, target_size, 
                                    context_len, mask_start, mask_end, mask_token_id):
    """
    Compute perplexity on target region with a masked window in context.
    
    Args:
        full_tokens: Full document tokens
        target_pos: Start of target region
        target_size: Size of target region
        context_len: Total context length (e.g., 512)
        mask_start: Start of mask within context (0-indexed from context start)
        mask_end: End of mask within context
        mask_token_id: Token ID to use for masking
    
    Returns:
        perplexity on target region
    """
    # Extract context + target
    context_start = target_pos - context_len
    if context_start < 0:
        return np.nan
    
    target_end = target_pos + target_size
    if target_end > len(full_tokens):
        return np.nan
    
    # Get tokens and apply mask
    tokens = list(full_tokens[context_start:target_end])
    
    # Mask is relative to context start
    for i in range(mask_start, min(mask_end, context_len)):
        tokens[i] = mask_token_id
    
    # Compute perplexity on target region
    target_start_local = context_len
    target_end_local = context_len + target_size
    
    ppl, _ = compute_perplexity_on_region(tokens, target_start_local, target_end_local)
    return ppl


def compute_anchor_profile(full_tokens, target_pos, target_size, context_len,
                           window_size, stride, mask_token_id):
    """
    Compute the context importance profile via sliding window ablation.
    
    Returns:
        window_starts: list of window start positions (relative to context)
        delta_profile: list of Δ values (ppl_masked - ppl_full)
        ppl_full: perplexity with full context (no mask)
    """
    # First compute baseline (no mask)
    context_start = target_pos - context_len
    if context_start < 0:
        return [], [], np.nan
    
    target_end = target_pos + target_size
    if target_end > len(full_tokens):
        return [], [], np.nan
    
    # Baseline perplexity
    tokens_full = full_tokens[context_start:target_end]
    ppl_full, _ = compute_perplexity_on_region(tokens_full, context_len, context_len + target_size)
    
    if np.isinf(ppl_full) or np.isnan(ppl_full):
        return [], [], np.nan
    
    # Sliding window ablation
    window_starts = []
    delta_profile = []
    
    for j in range(0, context_len - window_size + 1, stride):
        ppl_masked = compute_ppl_with_masked_window(
            full_tokens, target_pos, target_size,
            context_len, j, j + window_size, mask_token_id
        )
        
        if not np.isnan(ppl_masked) and not np.isinf(ppl_masked):
            delta_j = ppl_masked - ppl_full
            window_starts.append(j)
            delta_profile.append(delta_j)
    
    return window_starts, delta_profile, ppl_full


def compute_anchor_specificity_metrics(delta_profile):
    """
    Compute anchor specificity metrics from delta profile.
    
    Returns dict with:
        - top_share: max(Δ_pos) / sum(Δ_pos) - fraction from single best window
        - n_eff: effective number of windows (exp of entropy)
        - gini: Gini coefficient on positive impacts
        - n_positive: number of windows with positive impact
    """
    deltas = np.array(delta_profile)
    
    if len(deltas) == 0:
        return {
            'top_share': np.nan,
            'n_eff': np.nan,
            'gini': np.nan,
            'n_positive': 0,
            'max_delta': np.nan,
            'sum_delta_pos': np.nan,
        }
    
    # Only consider positive impacts (masking hurts prediction)
    deltas_pos = np.maximum(deltas, 0)
    sum_pos = np.sum(deltas_pos)
    n_positive = np.sum(deltas_pos > 0)
    
    if sum_pos <= 0:
        return {
            'top_share': 0,
            'n_eff': 0,
            'gini': np.nan,
            'n_positive': 0,
            'max_delta': np.max(deltas) if len(deltas) > 0 else np.nan,
            'sum_delta_pos': 0,
        }
    
    # Top share: how much of the benefit is from the single best window
    max_pos = np.max(deltas_pos)
    top_share = max_pos / sum_pos
    
    # Effective number of windows (entropy-based)
    p = deltas_pos / sum_pos
    p_nonzero = p[p > 0]
    if len(p_nonzero) > 0:
        entropy = -np.sum(p_nonzero * np.log(p_nonzero))
        n_eff = np.exp(entropy)
    else:
        n_eff = 0
    
    # Gini coefficient
    sorted_pos = np.sort(deltas_pos)
    n = len(sorted_pos)
    if n > 0 and sum_pos > 0:
        cumsum = np.cumsum(sorted_pos)
        gini = (2 * np.sum((np.arange(1, n+1) * sorted_pos)) - (n + 1) * sum_pos) / (n * sum_pos)
    else:
        gini = np.nan
    
    return {
        'top_share': top_share,
        'n_eff': n_eff,
        'gini': gini,
        'n_positive': int(n_positive),
        'max_delta': max_pos,
        'sum_delta_pos': sum_pos,
    }


print("Anchor specificity functions defined.")

In [ ]:
# ============================================================
# ANCHOR SPECIFICITY: Run Analysis
# ============================================================

# Get docs with valid I_512
anchor_eligible = metrics_df[metrics_df['burst_valid_512']].copy()
print(f"Docs eligible for anchor analysis: {len(anchor_eligible)}")

anchor_results = []
anchor_profiles = []  # Store full profiles for example plots

for idx, row in tqdm(anchor_eligible.iterrows(), total=len(anchor_eligible), 
                     desc="Anchor specificity"):
    doc_id = row['doc_id']
    
    # Get full tokens
    doc_row = df[df['doc_id'] == doc_id].iloc[0]
    full_tokens = tokenizer.encode(doc_row['text'])
    
    # Find t* = argmax(I_512)
    t_star, I_512_star, t_star_frac = find_max_I_512_target(traces_df, doc_id)
    
    if t_star is None:
        continue
    
    result = {
        'doc_id': doc_id,
        'age_months': row['age_months'],
        'age_group': row['age_group'],
        'n_tokens': row['n_tokens'],
        't_star': t_star,
        't_star_frac': t_star_frac,
        'I_512_star': I_512_star,
        'baseline_512': row['burst_baseline_512'],
    }
    
    # Compute anchor profile for L=512
    window_starts_512, delta_profile_512, ppl_full_512 = compute_anchor_profile(
        full_tokens, t_star, config.TARGET_REGION_TOKENS,
        anchor_config.CONTEXT_L, anchor_config.WINDOW_SIZE, 
        anchor_config.STRIDE, MASK_TOKEN_ID
    )
    
    if len(delta_profile_512) > 0:
        metrics_512 = compute_anchor_specificity_metrics(delta_profile_512)
        result['top_share_512'] = metrics_512['top_share']
        result['n_eff_512'] = metrics_512['n_eff']
        result['gini_512'] = metrics_512['gini']
        result['n_positive_512'] = metrics_512['n_positive']
        result['n_windows_512'] = len(delta_profile_512)
        result['ppl_full_512'] = ppl_full_512
        
        # Store profile for examples
        anchor_profiles.append({
            'doc_id': doc_id,
            'age_months': row['age_months'],
            'age_group': row['age_group'],
            'L': 512,
            'window_starts': window_starts_512,
            'delta_profile': delta_profile_512,
            't_star': t_star,
            'I_512_star': I_512_star,
        })
    else:
        result['top_share_512'] = np.nan
        result['n_eff_512'] = np.nan
        result['gini_512'] = np.nan
        result['n_positive_512'] = 0
        result['n_windows_512'] = 0
        result['ppl_full_512'] = np.nan
    
    # Also compute for L=256 if enabled
    if anchor_config.ALSO_DO_256 and t_star >= anchor_config.CONTEXT_S:
        window_starts_256, delta_profile_256, ppl_full_256 = compute_anchor_profile(
            full_tokens, t_star, config.TARGET_REGION_TOKENS,
            anchor_config.CONTEXT_S, anchor_config.WINDOW_SIZE,
            anchor_config.STRIDE, MASK_TOKEN_ID
        )
        
        if len(delta_profile_256) > 0:
            metrics_256 = compute_anchor_specificity_metrics(delta_profile_256)
            result['top_share_256'] = metrics_256['top_share']
            result['n_eff_256'] = metrics_256['n_eff']
            result['n_windows_256'] = len(delta_profile_256)
        else:
            result['top_share_256'] = np.nan
            result['n_eff_256'] = np.nan
            result['n_windows_256'] = 0
    else:
        result['top_share_256'] = np.nan
        result['n_eff_256'] = np.nan
        result['n_windows_256'] = 0
    
    anchor_results.append(result)

anchor_df = pd.DataFrame(anchor_results)

print(f"\n{'='*60}")
print("ANCHOR SPECIFICITY SUMMARY")
print(f"{'='*60}")
print(f"Documents analyzed: {len(anchor_df)}")
print(f"Valid L=512 profiles: {anchor_df['n_windows_512'].gt(0).sum()}")
if anchor_config.ALSO_DO_256:
    print(f"Valid L=256 profiles: {anchor_df['n_windows_256'].gt(0).sum()}")

In [ ]:
# ============================================================
# ANCHOR SPECIFICITY: Results by Age Group
# ============================================================

print("\n" + "=" * 70)
print("ANCHOR SPECIFICITY BY AGE GROUP")
print("=" * 70)

# Filter to valid results
anchor_valid = anchor_df[anchor_df['n_windows_512'] > 0].copy()
print(f"\nDocs with valid anchor profiles: {len(anchor_valid)}")

# Compute summary by age group
anchor_summary = []

for ag in age_groups:
    ag_df = anchor_valid[anchor_valid['age_group'] == ag]
    
    if len(ag_df) < 5:
        continue
    
    result = {'age_group': ag, 'n': len(ag_df)}
    
    # Top share (higher = more retrieval-like)
    mean, ci_low, ci_high = bootstrap_ci(
        ag_df['top_share_512'].values,
        n_bootstrap=config.N_BOOTSTRAP,
        ci=config.CI_LEVEL
    )
    result['top_share_512_mean'] = mean
    result['top_share_512_ci_low'] = ci_low
    result['top_share_512_ci_high'] = ci_high
    
    # N_eff (lower = more concentrated)
    mean, ci_low, ci_high = bootstrap_ci(
        ag_df['n_eff_512'].values,
        n_bootstrap=config.N_BOOTSTRAP,
        ci=config.CI_LEVEL
    )
    result['n_eff_512_mean'] = mean
    result['n_eff_512_ci_low'] = ci_low
    result['n_eff_512_ci_high'] = ci_high
    
    # I_512_star (max long-range influence)
    mean, ci_low, ci_high = bootstrap_ci(
        ag_df['I_512_star'].values,
        n_bootstrap=config.N_BOOTSTRAP,
        ci=config.CI_LEVEL
    )
    result['I_512_star_mean'] = mean
    result['I_512_star_ci_low'] = ci_low
    result['I_512_star_ci_high'] = ci_high
    
    # Also compute for L=256 if available
    ag_256 = ag_df[ag_df['n_windows_256'] > 0]
    if len(ag_256) >= 5:
        mean, ci_low, ci_high = bootstrap_ci(ag_256['top_share_256'].values)
        result['top_share_256_mean'] = mean
        result['n_256'] = len(ag_256)
        
        mean, ci_low, ci_high = bootstrap_ci(ag_256['n_eff_256'].values)
        result['n_eff_256_mean'] = mean
    
    anchor_summary.append(result)

anchor_summary_df = pd.DataFrame(anchor_summary)

print("\n--- TOP SHARE (higher = more retrieval-like) ---")
for _, row in anchor_summary_df.iterrows():
    print(f"{row['age_group']}: {row['top_share_512_mean']:.3f} "
          f"[{row['top_share_512_ci_low']:.3f}, {row['top_share_512_ci_high']:.3f}] "
          f"(n={row['n']})")

print("\n--- N_EFF (lower = more concentrated) ---")
for _, row in anchor_summary_df.iterrows():
    print(f"{row['age_group']}: {row['n_eff_512_mean']:.1f} "
          f"[{row['n_eff_512_ci_low']:.1f}, {row['n_eff_512_ci_high']:.1f}]")

print("\n--- MAX LONG-RANGE INFLUENCE (I_512 at t*) ---")
for _, row in anchor_summary_df.iterrows():
    print(f"{row['age_group']}: {row['I_512_star_mean']:.1f} "
          f"[{row['I_512_star_ci_low']:.1f}, {row['I_512_star_ci_high']:.1f}]")

# Compare L=512 vs L=256 top_share
if 'top_share_256_mean' in anchor_summary_df.columns:
    print("\n--- COMPARISON: top_share at L=512 vs L=256 ---")
    for _, row in anchor_summary_df.iterrows():
        if pd.notna(row.get('top_share_256_mean')):
            print(f"{row['age_group']}: L=512: {row['top_share_512_mean']:.3f}, "
                  f"L=256: {row['top_share_256_mean']:.3f}")

In [ ]:
# ============================================================
# ANCHOR SPECIFICITY: Statistical Tests
# ============================================================

print("\n" + "=" * 70)
print("ANCHOR SPECIFICITY STATISTICAL TESTS")
print("=" * 70)

print("\n--- Correlations with Age (Spearman) ---")
for col, label in [('top_share_512', 'Top share (retrieval-like)'),
                   ('n_eff_512', 'N_eff (# contributing windows)'),
                   ('I_512_star', 'Max I_512')]:
    valid = anchor_valid[[col, 'age_months']].dropna()
    if len(valid) > 10:
        rho, p = stats.spearmanr(valid['age_months'], valid[col])
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{label:<35}: rho={rho:+.3f}, p={p:.4f} {sig}")

print("\n--- Kruskal-Wallis (age group effect) ---")
for col, label in [('top_share_512', 'Top share'),
                   ('n_eff_512', 'N_eff')]:
    groups = [anchor_valid[anchor_valid['age_group'] == ag][col].dropna().values 
              for ag in age_groups if len(anchor_valid[anchor_valid['age_group'] == ag]) > 0]
    groups = [g for g in groups if len(g) >= 3]
    
    if len(groups) >= 2:
        h, p = stats.kruskal(*groups)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{label:<35}: H={h:.2f}, p={p:.4f} {sig}")

In [ ]:
# ============================================================
# ANCHOR SPECIFICITY: Example Profiles (Young vs Older)
# ============================================================

print("\n" + "=" * 70)
print("EXAMPLE ANCHOR PROFILES")
print("=" * 70)

# Find examples
young_profiles = [p for p in anchor_profiles if p['age_group'] == '4-6yr']
older_profiles = [p for p in anchor_profiles if p['age_group'] in ['8-10yr', '10+yr']]

if len(young_profiles) > 0 and len(older_profiles) > 0:
    # Pick examples with highest I_512_star
    young_ex = max(young_profiles, key=lambda x: x['I_512_star'])
    older_ex = max(older_profiles, key=lambda x: x['I_512_star'])
    
    # Get metrics for these docs
    young_metrics = anchor_valid[anchor_valid['doc_id'] == young_ex['doc_id']].iloc[0]
    older_metrics = anchor_valid[anchor_valid['doc_id'] == older_ex['doc_id']].iloc[0]
    
    print(f"\nYoung example: {young_ex['doc_id']} (age={young_ex['age_months']}mo)")
    print(f"  I_512 at t*: {young_ex['I_512_star']:.1f}")
    print(f"  top_share: {young_metrics['top_share_512']:.3f}")
    print(f"  n_eff: {young_metrics['n_eff_512']:.1f}")
    
    print(f"\nOlder example: {older_ex['doc_id']} (age={older_ex['age_months']}mo)")
    print(f"  I_512 at t*: {older_ex['I_512_star']:.1f}")
    print(f"  top_share: {older_metrics['top_share_512']:.3f}")
    print(f"  n_eff: {older_metrics['n_eff_512']:.1f}")
    
    # Plot profiles
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Young profile
    ax = axes[0]
    x = np.array(young_ex['window_starts'])
    y = np.array(young_ex['delta_profile'])
    ax.bar(x, y, width=anchor_config.STRIDE*0.9, 
           color=['#e74c3c' if v > 0 else '#95a5a6' for v in y], alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Window start position (tokens before target)')
    ax.set_ylabel('Δ perplexity (masked - full)')
    ax.set_title(f"Young ({young_ex['age_months']}mo): Context importance\n"
                 f"top_share={young_metrics['top_share_512']:.3f}, n_eff={young_metrics['n_eff_512']:.1f}")
    ax.set_xlim(-10, anchor_config.CONTEXT_L + 10)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add annotation for peak
    if len(y) > 0:
        peak_idx = np.argmax(y)
        ax.annotate(f'peak @ {x[peak_idx]}', xy=(x[peak_idx], y[peak_idx]),
                    xytext=(x[peak_idx]+50, y[peak_idx]+1),
                    arrowprops=dict(arrowstyle='->', color='black'),
                    fontsize=9)
    
    # Older profile
    ax = axes[1]
    x = np.array(older_ex['window_starts'])
    y = np.array(older_ex['delta_profile'])
    ax.bar(x, y, width=anchor_config.STRIDE*0.9,
           color=['#27ae60' if v > 0 else '#95a5a6' for v in y], alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Window start position (tokens before target)')
    ax.set_ylabel('Δ perplexity (masked - full)')
    ax.set_title(f"Older ({older_ex['age_months']}mo): Context importance\n"
                 f"top_share={older_metrics['top_share_512']:.3f}, n_eff={older_metrics['n_eff_512']:.1f}")
    ax.set_xlim(-10, anchor_config.CONTEXT_L + 10)
    ax.grid(True, alpha=0.3, axis='y')
    
    if len(y) > 0:
        peak_idx = np.argmax(y)
        ax.annotate(f'peak @ {x[peak_idx]}', xy=(x[peak_idx], y[peak_idx]),
                    xytext=(x[peak_idx]+50, y[peak_idx]+1),
                    arrowprops=dict(arrowstyle='->', color='black'),
                    fontsize=9)
    
    plt.tight_layout()
    plt.savefig('ecsc_anchor_example_profiles.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Not enough examples for profile comparison.")

In [ ]:
# ============================================================
# ANCHOR SPECIFICITY: Summary Visualization
# ============================================================

if len(anchor_summary_df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Anchor Specificity: Retrieval-like vs Distributed Influence', 
                 fontsize=14, fontweight='bold')
    
    x_pos = range(len(anchor_summary_df))
    
    # 1. Top share by age group
    ax = axes[0, 0]
    means = anchor_summary_df['top_share_512_mean'].values
    ci_low = anchor_summary_df['top_share_512_ci_low'].values
    ci_high = anchor_summary_df['top_share_512_ci_high'].values
    errors = [means - ci_low, ci_high - means]
    
    bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
                  color=[colors.get(ag, 'gray') for ag in anchor_summary_df['age_group']],
                  alpha=0.7, edgecolor='black')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"{row['age_group']}\n(n={row['n']})" 
                        for _, row in anchor_summary_df.iterrows()])
    ax.set_ylabel('Top Share')
    ax.set_title('Top Share (higher = more retrieval-like)')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 2. N_eff by age group
    ax = axes[0, 1]
    means = anchor_summary_df['n_eff_512_mean'].values
    ci_low = anchor_summary_df['n_eff_512_ci_low'].values
    ci_high = anchor_summary_df['n_eff_512_ci_high'].values
    errors = [means - ci_low, ci_high - means]
    
    bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
                  color=[colors.get(ag, 'gray') for ag in anchor_summary_df['age_group']],
                  alpha=0.7, edgecolor='black')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"{row['age_group']}" for _, row in anchor_summary_df.iterrows()])
    ax.set_ylabel('N_eff (effective windows)')
    ax.set_title('N_eff (lower = more concentrated)')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 3. Max I_512 by age group
    ax = axes[0, 2]
    means = anchor_summary_df['I_512_star_mean'].values
    ci_low = anchor_summary_df['I_512_star_ci_low'].values
    ci_high = anchor_summary_df['I_512_star_ci_high'].values
    errors = [means - ci_low, ci_high - means]
    
    bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
                  color=[colors.get(ag, 'gray') for ag in anchor_summary_df['age_group']],
                  alpha=0.7, edgecolor='black')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"{row['age_group']}" for _, row in anchor_summary_df.iterrows()])
    ax.set_ylabel('Max I_512')
    ax.set_title('Peak Long-Range Influence')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 4. Scatter: age vs top_share
    ax = axes[1, 0]
    for ag in age_groups:
        ag_data = anchor_valid[anchor_valid['age_group'] == ag]
        if len(ag_data) > 0:
            ax.scatter(ag_data['age_months'], ag_data['top_share_512'],
                       alpha=0.5, label=ag, color=colors.get(ag, 'gray'), s=30)
    
    valid = anchor_valid[['age_months', 'top_share_512']].dropna()
    if len(valid) > 10:
        z = np.polyfit(valid['age_months'], valid['top_share_512'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(valid['age_months'].min(), valid['age_months'].max(), 100)
        ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2)
    
    ax.set_xlabel('Age (months)')
    ax.set_ylabel('Top Share')
    ax.set_title('Age vs Top Share')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 5. Scatter: age vs n_eff
    ax = axes[1, 1]
    for ag in age_groups:
        ag_data = anchor_valid[anchor_valid['age_group'] == ag]
        if len(ag_data) > 0:
            ax.scatter(ag_data['age_months'], ag_data['n_eff_512'],
                       alpha=0.5, label=ag, color=colors.get(ag, 'gray'), s=30)
    
    valid = anchor_valid[['age_months', 'n_eff_512']].dropna()
    if len(valid) > 10:
        z = np.polyfit(valid['age_months'], valid['n_eff_512'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(valid['age_months'].min(), valid['age_months'].max(), 100)
        ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2)
    
    ax.set_xlabel('Age (months)')
    ax.set_ylabel('N_eff')
    ax.set_title('Age vs N_eff')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 6. Scatter: top_share vs n_eff (should be inversely related)
    ax = axes[1, 2]
    for ag in age_groups:
        ag_data = anchor_valid[anchor_valid['age_group'] == ag]
        if len(ag_data) > 0:
            ax.scatter(ag_data['top_share_512'], ag_data['n_eff_512'],
                       alpha=0.5, label=ag, color=colors.get(ag, 'gray'), s=30)
    
    ax.set_xlabel('Top Share')
    ax.set_ylabel('N_eff')
    ax.set_title('Top Share vs N_eff\n(inversely related)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('ecsc_anchor_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Not enough data for anchor summary visualization.")

In [ ]:
# ============================================================
# ANCHOR SPECIFICITY: Save Results to Google Drive
# ============================================================

anchor_df['model'] = MODEL_DISPLAY
anchor_df['model_tag'] = MODEL_TAG
anchor_df.to_csv(f'{VERSION_DIR}/ecsc_{VERSION}_{MODEL_TAG}_anchor.csv', index=False)

anchor_summary_df['model'] = MODEL_DISPLAY
anchor_summary_df.to_csv(f'{VERSION_DIR}/ecsc_{VERSION}_{MODEL_TAG}_anchor_summary.csv', index=False)

import shutil
for fig_name in ['ecsc_anchor_example_profiles.png', 'ecsc_anchor_summary.png']:
    if os.path.exists(fig_name):
        new_name = fig_name.replace('ecsc_', f'ecsc_{MODEL_TAG}_')
        shutil.copy(fig_name, f'{FIGURES_DIR}/{new_name}')

print(f"Anchor results saved: ecsc_{VERSION}_{MODEL_TAG}_anchor*.csv")

In [ ]:
# ============================================================
# UPDATE MASTER RESULTS (auto-append to Drive)
# ============================================================

MASTER_RESULTS_CODE = '''
import os
import re
from datetime import datetime
import numpy as np
import pandas as pd

STANDARD_AGE_BINS = ["4-6", "6-8", "8-10", "10+"]
AGE_BIN_ORDER = {b: i for i, b in enumerate(STANDARD_AGE_BINS)}
AGE_BIN_NORMALIZE = {
    "4-6yr": "4-6", "6-8yr": "6-8", "8-10yr": "8-10", "10+yr": "10+",
    "4-6": "4-6", "6-8": "6-8", "8-10": "8-10", "10+": "10+",
}

def normalize_age_bin(age_bin):
    return AGE_BIN_NORMALIZE.get(age_bin, age_bin)

def extract_endofdoc_metrics(df, age_bin):
    metrics = {
        "half_life_512_mean": np.nan, "half_life_512_ci_low": np.nan, "half_life_512_ci_high": np.nan,
        "n_endofdoc": 0,
        "early_drop_pct_512_mean": np.nan, "early_drop_pct_512_ci_low": np.nan, "early_drop_pct_512_ci_high": np.nan,
        "ppl_min_ctx_mean": np.nan, "ppl_min_ctx_ci_low": np.nan, "ppl_min_ctx_ci_high": np.nan,
    }
    if df is None or len(df) == 0:
        return metrics
    mask = df["age_group"].apply(normalize_age_bin) == age_bin
    if not mask.any():
        return metrics
    row = df[mask].iloc[0]
    col_map = {
        "half_life_512_mean": ["half_life_fixed_mean"], "half_life_512_ci_low": ["half_life_fixed_ci_low"],
        "half_life_512_ci_high": ["half_life_fixed_ci_high"], "n_endofdoc": ["n"],
        "early_drop_pct_512_mean": ["early_drop_pct_fixed_mean"],
        "early_drop_pct_512_ci_low": ["early_drop_pct_fixed_ci_low"],
        "early_drop_pct_512_ci_high": ["early_drop_pct_fixed_ci_high"],
        "ppl_min_ctx_mean": ["ppl_min_ctx_mean"], "ppl_min_ctx_ci_low": ["ppl_min_ctx_ci_low"],
        "ppl_min_ctx_ci_high": ["ppl_min_ctx_ci_high"],
    }
    for out_col, in_cols in col_map.items():
        for in_col in in_cols:
            if in_col in row.index:
                metrics[out_col] = row[in_col]
                break
    return metrics

def extract_burst_metrics(df, age_bin, L):
    prefix = f"b{L}_"
    metrics = {
        f"{prefix}baseline_mean": np.nan, f"{prefix}baseline_ci_low": np.nan, f"{prefix}baseline_ci_high": np.nan,
        f"{prefix}n_valid": 0, 
        f"{prefix}spike_mass_c_mean": np.nan, f"{prefix}spike_mass_c_ci_low": np.nan, f"{prefix}spike_mass_c_ci_high": np.nan,
    }
    if df is None or len(df) == 0:
        return metrics
    mask = df["age_group"].apply(normalize_age_bin) == age_bin
    if not mask.any():
        return metrics
    row = df[mask].iloc[0]
    for col in ["baseline_mean", "baseline_ci_low", "baseline_ci_high", 
                "spike_mass_c_mean", "spike_mass_c_ci_low", "spike_mass_c_ci_high"]:
        if col in row.index:
            metrics[f"{prefix}{col}"] = row[col]
    if "n" in row.index:
        metrics[f"{prefix}n_valid"] = int(row["n"])
    return metrics

def extract_anchor_metrics(df, age_bin):
    metrics = {
        "a512_top_share_mean": np.nan, "a512_top_share_ci_low": np.nan, "a512_top_share_ci_high": np.nan,
        "a512_n_eff_mean": np.nan, "a512_n_eff_ci_low": np.nan, "a512_n_eff_ci_high": np.nan,
        "a512_n_valid": 0,
    }
    if df is None or len(df) == 0:
        return metrics
    mask = df["age_group"].apply(normalize_age_bin) == age_bin
    if not mask.any():
        return metrics
    row = df[mask].iloc[0]
    col_map = {
        "a512_top_share_mean": ["top_share_512_mean", "top_share_mean"],
        "a512_top_share_ci_low": ["top_share_512_ci_low", "top_share_ci_low"],
        "a512_top_share_ci_high": ["top_share_512_ci_high", "top_share_ci_high"],
        "a512_n_eff_mean": ["n_eff_512_mean", "n_eff_mean"],
        "a512_n_eff_ci_low": ["n_eff_512_ci_low", "n_eff_ci_low"],
        "a512_n_eff_ci_high": ["n_eff_512_ci_high", "n_eff_ci_high"],
        "a512_n_valid": ["n"],
    }
    for out_col, in_cols in col_map.items():
        for in_col in in_cols:
            if in_col in row.index:
                val = row[in_col]
                metrics[out_col] = int(val) if out_col.endswith("n_valid") else val
                break
    return metrics
'''

exec(MASTER_RESULTS_CODE)

# Get model info
_model_name = MODEL_NAME
_model_tag = MODEL_TAG
_quantization = QUANTIZATION

run_prefix = f"ecsc_{VERSION}_{_model_tag}"

# Paths
master_dir = f"{DRIVE_ROOT}/results"
os.makedirs(master_dir, exist_ok=True)
master_runs_path = f"{master_dir}/master_runs.csv"
master_agebin_path = f"{master_dir}/master_agebin.csv"

# Build run_id with YYYYMMDD_HHMM to avoid collisions
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
run_id = f"{run_prefix}_{timestamp}"

# Artifact paths (relative to DRIVE_ROOT for portability)
artifacts_reldir = f"results/ecsc/{VERSION}_{_model_tag}"
artifact_paths = {
    "endofdoc_summary_path": f"{artifacts_reldir}/ecsc_{VERSION}_{_model_tag}_endofdoc_summary.csv",
    "burst256_path": f"{artifacts_reldir}/ecsc_{VERSION}_{_model_tag}_burst_summary_256.csv",
    "burst512_path": f"{artifacts_reldir}/ecsc_{VERSION}_{_model_tag}_burst_summary_512.csv",
    "anchor_summary_path": f"{artifacts_reldir}/ecsc_{VERSION}_{_model_tag}_anchor_summary.csv",
    "anchor_path": f"{artifacts_reldir}/ecsc_{VERSION}_{_model_tag}_anchor.csv",
    "metrics_path": f"{artifacts_reldir}/ecsc_{VERSION}_{_model_tag}_metrics.csv",
    "config_path": f"{artifacts_reldir}/ecsc_{VERSION}_{_model_tag}_config.json",
}

# Sample sizes
n_docs_total = len(metrics_df)
n_docs_burst256 = int(metrics_df["burst_valid_256"].sum()) if "burst_valid_256" in metrics_df.columns else 0
n_docs_burst512 = int(metrics_df["burst_valid_512"].sum()) if "burst_valid_512" in metrics_df.columns else 0
n_docs_anchor = len(anchor_df) if "anchor_df" in dir() and anchor_df is not None else 0

# Run row with artifact paths
run_row = {
    "run_id": run_id, 
    "dataset": "ECSC", 
    "version": VERSION,
    "model_name": _model_name, 
    "model_tag": _model_tag,
    "precision": _quantization,
    "k_targets": config.K_TARGETS, 
    "min_words": config.MIN_WORDS,
    "short_context": config.SHORT_CONTEXT,
    "long_contexts": str(config.LONG_CONTEXTS),
    "date_run": datetime.now().isoformat(),
    "n_docs_total": n_docs_total, 
    "n_docs_burst256_valid": n_docs_burst256,
    "n_docs_burst512_valid": n_docs_burst512, 
    "n_docs_anchor512_valid": n_docs_anchor,
    **artifact_paths,
}

# Age bin rows with n_valid per bin
age_bins = set(STANDARD_AGE_BINS)
agebin_rows = []
for age_bin in sorted(age_bins, key=lambda x: AGE_BIN_ORDER.get(x, 99)):
    row = {
        "run_id": run_id, 
        "dataset": "ECSC", 
        "version": VERSION,
        "model_name": _model_name, 
        "model_tag": _model_tag,
        "precision": _quantization, 
        "age_bin": age_bin,
        "age_bin_order": AGE_BIN_ORDER.get(age_bin, 99),
    }
    row.update(extract_endofdoc_metrics(endofdoc_summary_df, age_bin))
    row.update(extract_burst_metrics(burst_summaries.get(256), age_bin, 256))
    row.update(extract_burst_metrics(burst_summaries.get(512), age_bin, 512))
    if "anchor_summary_df" in dir() and anchor_summary_df is not None:
        row.update(extract_anchor_metrics(anchor_summary_df, age_bin))
    else:
        row.update({"a512_top_share_mean": np.nan, "a512_n_eff_mean": np.nan, "a512_n_valid": 0})
    agebin_rows.append(row)

# Load existing and merge (replace runs with same prefix)
if os.path.exists(master_runs_path):
    existing_runs = pd.read_csv(master_runs_path)
    existing_runs = existing_runs[~existing_runs["run_id"].str.startswith(run_prefix + "_")]
    runs_df = pd.concat([existing_runs, pd.DataFrame([run_row])], ignore_index=True)
else:
    runs_df = pd.DataFrame([run_row])

if os.path.exists(master_agebin_path):
    existing_agebin = pd.read_csv(master_agebin_path)
    existing_agebin = existing_agebin[~existing_agebin["run_id"].str.startswith(run_prefix + "_")]
    agebin_df_master = pd.concat([existing_agebin, pd.DataFrame(agebin_rows)], ignore_index=True)
else:
    agebin_df_master = pd.DataFrame(agebin_rows)

# Sort and save
runs_df = runs_df.sort_values(["dataset", "model_tag", "date_run"])
agebin_df_master = agebin_df_master.sort_values(["dataset", "model_tag", "age_bin_order"])

runs_df.to_csv(master_runs_path, index=False)
agebin_df_master.to_csv(master_agebin_path, index=False)

print(f"\n{'='*60}")
print("MASTER RESULTS UPDATED")
print(f"{'='*60}")
print(f"Run: {run_id}")
print(f"  n_docs: {n_docs_total}, burst512: {n_docs_burst512}, anchor: {n_docs_anchor}")
print(f"\nAgebin n_valid (b512 / a512):")
for r in agebin_rows:
    print(f"  {r['age_bin']}: b512={r.get('b512_n_valid', 0)} / a512={r.get('a512_n_valid', 0)}")
print(f"\nMaster files ({master_dir}):")
print(f"  master_runs.csv: {len(runs_df)} runs")
print(f"  master_agebin.csv: {len(agebin_df_master)} rows")
print(f"\nArtifact paths recorded in master_runs.csv")